# Gold dimension -- `dbo.dim_institution`

Companies occupying space in the buildings. 30 rows. This is the dimension where type1 costs the most: a tenant relocating silently re-attributes all of their past arrivals to the new building.

**Source:** `silver.stg_institution`  
**SCD:** `type1`  
**Surrogate key:** `institution_sk`  
**Tracked columns:** `n/a`

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_silver, writes to wh_gold. Both must be
# attached to this notebook; wh_gold must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "wh_gold"
source_item = "lh_silver"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="dim_institution",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="gold", table_name="dim_institution")
print(f"load_id={load_id}  environment={environment}  table=dim_institution")

from ttfabric.warehouse import gold_target

gold = gold_target(
    spark,
    warehouse="wh_gold",
    schema="dbo",
    write_mode="warehouse_connector",
)


In [ ]:
# ---- Read silver -------------------------------------------------
src = spark.read.table(f"{source_item}.stg_institution")
print(f"read {src.count():,} rows from stg_institution")


In [ ]:
# ---- Project to the target schema --------------------------------
# Renames come from mappings/gold.yaml `columns`. Applied before the
# business rules, which are written against target names.
src = src.select(
    F.col("institution_id"),
    F.col("institution_name"),
    F.col("building_key"),
)


In [ ]:
# ---- SCD type 1 overwrite ----------------------------------------
from ttfabric.dimensions import assign_surrogate_key

out = assign_surrogate_key(src, "institution_sk", ['institution_id'])
gold.write(out, "dim_institution")


In [ ]:
# ---- Unknown member ----------------------------------------------
# Guarantees an unmatched fact still joins rather than vanishing
# from a report without trace.
from ttfabric.dimensions import ensure_unknown_member

ensure_unknown_member(
    spark,
    table="dim_institution",
    surrogate_key="institution_sk",
    key_value=-1,
    defaults={'institution_id': -1, 'institution_name': 'Unknown', 'building_key': -1},
    gold=gold,
)
dq.flush()
